In [ ]:
#import sklearn
import igraph
import harmony
import anndata as ad
#import scanpy as sc
import pegasus as pg
import pandas as pd
import os
from scipy import sparse, io 
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sb
import matplotlib
import matplotlib.colors as col
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
cmap2 = col.LinearSegmentedColormap.from_list('own',['#A0A0A0','#000000','red','orange'])
cm.register_cmap(cmap=cmap2)

In [ ]:
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42

In [ ]:
data = pg.read_input('/cluster3/yflu/STS/TMA/STS_TMA_counts_20250730.h5ad')

In [ ]:
data.obs.index

In [ ]:
samples = data.obs["Sample"].unique()

In [ ]:
samples = pd.Series(samples).loc[lambda x: ~x.isin(["T491","T1113","T356","T253","T365","T1668"])].astype("category")
samples = pd.Series(samples, dtype="category").cat.remove_categories(["T491","T1113","T356","T253","T365","T1668"])

In [ ]:
samples

In [ ]:
# List to store individual DataFrames
dfs = []

# Iterate over each sample in the list
for sample in samples:
    # Construct the file path dynamically
    file_path = f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_celltype.csv"
    
    # Read the CSV and append to the list
    df = pd.read_csv(file_path, sep=',', header=0)
    mask = data.obs["Sample"] == sample
    matching_indices = data.obs[mask].index
    integer_positions = np.where(mask)[0]
    index = integer_positions[0]
    TMA = data.obs["Channel"][index]
    df['barcode'] = TMA + "-" + df['barcode'].str[:-2]
    dfs.append(df)

# Combine all DataFrames vertically (row-wise)
combined_df = pd.concat(dfs, axis=0, ignore_index=True)

In [ ]:
combined_df

In [ ]:
data_1=data[data.obs.index.isin(combined_df["barcode"])].copy()

In [ ]:
data_1

In [ ]:
data_1.obs.index

In [ ]:
df_aligned = combined_df.set_index("barcode").reindex(data.obs.index)

In [ ]:
df_aligned

In [ ]:
data_1.obs["celltype"] = df_aligned["celltype"]

In [ ]:
data_1.obs["celltype"].value_counts()

In [ ]:
pg.identify_robust_genes(data_1)
pg.log_norm(data_1,norm_count = 5000)
pg.calc_signature_score(data_1, 'cell_cycle_human')
pg.pca(data_1)
pg.regress_out(data_1, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_1,rep = 'pca_regressed',batch = "Sample")
pg.neighbors(data_1,rep=pca_key)
pg.louvain(data_1,rep=pca_key,class_label='louvain_labels',resolution=0.8)
pg.umap(data_1,rep=pca_key,out_basis='umap')
pg.tsne(data_1,rep=pca_key,out_basis='tsne')

In [ ]:
pg.scatter(data_1, ["celltype"])

In [ ]:
data_1.obs.index

In [ ]:
data_1

In [ ]:
data_1.X.min()

In [ ]:
adata = data_1.to_anndata()

In [ ]:
pg.violin(adata,attrs=['CD3E'], groupby='louvain_labels')

In [ ]:
data_1

In [ ]:
adata.write("/cluster3/yflu/STS/TMA/TMA_pegasus_250826_adata.h5ad")

In [ ]:
data_tumor = data_1[data_1.obs.celltype.isin(["Tumor cells"])].copy()

In [ ]:
data_normal = data_1[~data_1.obs.celltype.isin(["Tumor cells"])].copy()

In [ ]:
data_normal.obs["celltype"].value_counts()

In [ ]:
pg.identify_robust_genes(data_tumor)
pg.log_norm(data_tumor,norm_count = 5000)
pg.calc_signature_score(data_tumor, 'cell_cycle_human')
pg.pca(data_tumor)
pg.regress_out(data_tumor, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_tumor,rep = 'pca_regressed',batch = "Sample")
pg.neighbors(data_tumor,rep=pca_key)
pg.louvain(data_tumor,rep=pca_key,class_label='louvain_labels',resolution=0.8)
pg.umap(data_tumor,rep=pca_key,out_basis='umap')
pg.tsne(data_tumor,rep=pca_key,out_basis='tsne')

In [ ]:
pg.identify_robust_genes(data_normal)
pg.log_norm(data_normal,norm_count = 5000)
pg.calc_signature_score(data_normal, 'cell_cycle_human')
pg.pca(data_normal)
pg.regress_out(data_normal, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_normal,rep = 'pca_regressed',batch = "Sample")
pg.neighbors(data_normal,rep=pca_key)
pg.louvain(data_normal,rep=pca_key,class_label='louvain_labels',resolution=0.8)
pg.umap(data_normal,rep=pca_key,out_basis='umap')
pg.tsne(data_normal,rep=pca_key,out_basis='tsne')

In [ ]:
pg.write_output(data_tumor,"/cluster3/yflu/STS/TMA/TMA_tumor_pegasus_250816.h5ad")
pg.write_output(data_normal,"/cluster3/yflu/STS/TMA/TMA_normal_pegasus_250816.h5ad")

In [ ]:
data_normal = pg.read_input("/cluster3/yflu/STS/TMA/TMA_normal_pegasus_250816.h5ad")

In [ ]:
data_tumor = pg.read_input("/cluster3/yflu/STS/TMA/TMA_tumor_pegasus_250816.h5ad")

In [ ]:
pg.scatter(data_tumor, ["louvain_labels"],legend_loc = "on data")

In [ ]:
pg.scatter(data_normal, ["louvain_labels"],legend_loc = "on data")

In [ ]:
help(pg.scatter)

In [ ]:
pg.scatter(data_normal, ["POSTN","COL5A1","PECAM1","RGS5","CD3E","CD79A","CD68"])

In [ ]:
pg.scatter(data_normal, ["ADIPOQ"])
pg.violin(data_normal, ["ADIPOQ"],groupby = "louvain_labels")

In [ ]:
data_normal.obs['celltype']=data_normal.obs.louvain_labels.copy()
data_normal.obs.celltype.replace(['1'],'Fibroblasts',inplace=True)
data_normal.obs.celltype.replace(['2'],'Fibroblasts',inplace=True)
data_normal.obs.celltype.replace(['3'],'Mono/macro',inplace=True)
data_normal.obs.celltype.replace(['4'],'Endothelial',inplace=True)
data_normal.obs.celltype.replace(['5'],'Pericytes',inplace=True)
data_normal.obs.celltype.replace(['6'],'Mono/macro',inplace=True)
data_normal.obs.celltype.replace(['7'],'T cells',inplace=True)
data_normal.obs.celltype.replace(['8'],'Fibroblasts',inplace=True)
data_normal.obs.celltype.replace(['9'],'Plasma cells',inplace=True)
data_normal.obs.celltype.replace(['10'],'B cells',inplace=True)
data_normal.obs.celltype.replace(['11'],'DC',inplace=True)
data_normal.obs.celltype.replace(['12'],'Mono/macro',inplace=True)
data_normal.obs.celltype.replace(['13'],'Mast cells',inplace=True)
data_normal.obs.celltype.replace(['14'],'Hepatocytes',inplace=True)
data_normal.obs.celltype.replace(['15'],'Liver sinusoidal endothelial cells',inplace=True)
data_normal.obs.celltype.replace(['16'],'Fibroblasts',inplace=True)
data_normal.obs.celltype.replace(['17'],'Smooth muscle cells',inplace=True)
data_normal.obs.celltype.replace(['18'],'Smooth muscle cells',inplace=True)
data_normal.obs.celltype.replace(['19'],'Neural progenitors',inplace=True)
data_normal.obs.celltype.replace(['20'],'Adipocytes',inplace=True)

In [ ]:
pg.write_output(data_normal,"/cluster3/yflu/STS/TMA/TMA_normal_pegasus_250823.h5ad")

In [ ]:
data_normal = pg.read_input("/cluster3/yflu/STS/TMA/TMA_normal_pegasus_250823.h5ad")

In [ ]:
pg.scatter(data_normal, ["celltype"],basis = "tsne")

In [ ]:
data_T = data_normal[data_normal.obs.celltype.isin(["T cells"])].copy()
data_B = data_normal[data_normal.obs.celltype.isin(["B cells","Plasma cells"])].copy()
data_M = data_normal[data_normal.obs.celltype.isin(["Mono/macro","DC","Mast cells"])].copy()

In [ ]:
pg.scatter(data_M, ["CD68"],legend_loc = "on data")

In [ ]:
pg.identify_robust_genes(data_T)
pg.log_norm(data_T,norm_count = 5000)
pg.calc_signature_score(data_T, 'cell_cycle_human')
pg.pca(data_T)
pg.regress_out(data_T, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_T,rep = 'pca_regressed',batch = "Sample")
pg.neighbors(data_T,rep=pca_key)
pg.louvain(data_T,rep=pca_key,class_label='louvain_labels',resolution=0.8)
pg.umap(data_T,rep=pca_key,out_basis='umap')
pg.tsne(data_T,rep=pca_key,out_basis='tsne')

pg.identify_robust_genes(data_B)
pg.log_norm(data_B,norm_count = 5000)
pg.calc_signature_score(data_B, 'cell_cycle_human')
pg.pca(data_B)
pg.regress_out(data_B, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_B,rep = 'pca_regressed',batch = "Sample")
pg.neighbors(data_B,rep=pca_key)
pg.louvain(data_B,rep=pca_key,class_label='louvain_labels',resolution=0.8)
pg.umap(data_B,rep=pca_key,out_basis='umap')
pg.tsne(data_B,rep=pca_key,out_basis='tsne')

pg.identify_robust_genes(data_M)
pg.log_norm(data_M,norm_count = 5000)
pg.calc_signature_score(data_M, 'cell_cycle_human')
pg.pca(data_M)
pg.regress_out(data_M, attrs=['G1/S', 'G2/M'])
pca_key = pg.run_harmony(data_M,rep = 'pca_regressed',batch = "Sample")
pg.neighbors(data_M,rep=pca_key)
pg.louvain(data_M,rep=pca_key,class_label='louvain_labels',resolution=0.8)
pg.umap(data_M,rep=pca_key,out_basis='umap')
pg.tsne(data_M,rep=pca_key,out_basis='tsne')

In [ ]:
pg.write_output(data_T,"/cluster3/yflu/STS/TMA/TMA_T_pegasus_250820.h5ad")
pg.write_output(data_B,"/cluster3/yflu/STS/TMA/TMA_B_pegasus_250820.h5ad")
pg.write_output(data_M,"/cluster3/yflu/STS/TMA/TMA_M_pegasus_250820.h5ad")

In [ ]:
pca_key = "pca_regressed_harmony"
pg.neighbors(data_M,rep=pca_key,K=20)
pg.louvain(data_M,rep=pca_key,class_label='louvain_labels',resolution=0.8)

In [ ]:
pg.louvain(data_M,rep=pca_key,class_label='louvain_labels_1',resolution=1)
pg.louvain(data_M,rep=pca_key,class_label='louvain_labels_12',resolution=1.2)

In [ ]:
pca_key = "pca_regressed_harmony"
pg.louvain(data_T,rep=pca_key,class_label='louvain_labels_1',resolution=1)
pg.louvain(data_T,rep=pca_key,class_label='louvain_labels_12',resolution=1.2)
pg.louvain(data_B,rep=pca_key,class_label='louvain_labels_1',resolution=1)
pg.louvain(data_B,rep=pca_key,class_label='louvain_labels_12',resolution=1.2)
pg.louvain(data_M,rep=pca_key,class_label='louvain_labels_1',resolution=1)
pg.louvain(data_M,rep=pca_key,class_label='louvain_labels_12',resolution=1.2)

In [ ]:
pg.write_output(data_T,"/cluster3/yflu/STS/TMA/TMA_T_pegasus_250820.h5ad")
pg.write_output(data_B,"/cluster3/yflu/STS/TMA/TMA_B_pegasus_250820.h5ad")
pg.write_output(data_M,"/cluster3/yflu/STS/TMA/TMA_M_pegasus_250820.h5ad")

In [ ]:
pg.de_analysis(data_T, cluster='louvain_labels_1')
marker_dict_T = pg.markers(data_T)
pg.write_results_to_excel(marker_dict_T, "TMA_T_de.xlsx")
pg.de_analysis(data_B, cluster='louvain_labels_1')
marker_dict_B = pg.markers(data_B)
pg.write_results_to_excel(marker_dict_B, "TMA_B_de.xlsx")

In [ ]:
pg.de_analysis(data_M, cluster='louvain_labels')
marker_dict_M = pg.markers(data_M)
pg.write_results_to_excel(marker_dict_M, "TMA_M_de.xlsx")

In [ ]:
data_M = pg.read_input("/cluster3/yflu/STS/TMA/TMA_M_pegasus_250820.h5ad")

In [ ]:
data_T = pg.read_input("/cluster3/yflu/STS/TMA/TMA_T_pegasus_250820.h5ad")
data_B = pg.read_input("/cluster3/yflu/STS/TMA/TMA_B_pegasus_250820.h5ad")
data_M = pg.read_input("/cluster3/yflu/STS/TMA/TMA_M_pegasus_250822.h5ad")

In [ ]:
pg.de_analysis(data_M, cluster='louvain_labels')
marker_dict_M = pg.markers(data_M)
pg.write_results_to_excel(marker_dict_M, "TMA_M_de.xlsx")

In [ ]:
pg.write_output(data_M,"/cluster3/yflu/STS/TMA/TMA_M_pegasus_250822.h5ad")

In [ ]:
help(pg.neighbors)

In [ ]:
pg.scatter(data_B, ["louvain_labels_1"],legend_loc = "on data")

In [ ]:
pg.scatter(data_B, ["XBP1"])
pg.violin(data_B, ["CD14"],groupby = "louvain_labels_1")

In [ ]:
data_B.obs['celltype_sp']=data_B.obs.louvain_labels_1.copy()
data_B.obs.celltype_sp.replace(['1'],'Plasma cells',inplace=True)
data_B.obs.celltype_sp.replace(['2'],'Plasma cells',inplace=True)
data_B.obs.celltype_sp.replace(['3'],'Plasma cells fibro_niche',inplace=True)
data_B.obs.celltype_sp.replace(['4'],'B cells',inplace=True)
data_B.obs.celltype_sp.replace(['5'],'Plasma cells',inplace=True)
data_B.obs.celltype_sp.replace(['6'],'B cells mye_niche',inplace=True)
data_B.obs.celltype_sp.replace(['7'],'Plasma cells endo_niche',inplace=True)

In [ ]:
pg.scatter(data_B, ["celltype_sp"])

In [ ]:
pg.write_output(data_B,"/cluster3/yflu/STS/TMA/TMA_B_pegasus_250822.h5ad")

In [ ]:
import shutil

In [ ]:
shutil.copy("/cluster3/yflu/STS/TMA/TMA_T_pegasus_250820.h5ad","/cluster3/yflu/STS/TMA/normal/T/sct_spatial.h5ad")

In [ ]:
shutil.copy("/cluster3/yflu/STS/TMA/TMA_B_pegasus_250820.h5ad","/cluster3/yflu/STS/TMA/normal/B/sct_spatial.h5ad")
shutil.copy("/cluster3/yflu/STS/TMA/TMA_M_pegasus_250820.h5ad","/cluster3/yflu/STS/TMA/normal/M/sct_spatial.h5ad")

In [ ]:
pg.scatter(data_T, ["IL7R"])

In [ ]:
adata = pg.read_input(f'/cluster3/yflu/STS/TMA/normal/normal_nico.h5ad')

In [ ]:
data_T.obs['nico_ct'] = adata.obs.loc[data_T.obs_names, 'nico_ct']
data_B.obs['nico_ct'] = adata.obs.loc[data_B.obs_names, 'nico_ct']
data_M.obs['nico_ct'] = adata.obs.loc[data_M.obs_names, 'nico_ct']

In [ ]:
data_T.obs['nico_ct'] = data_T.obs['nico_ct'].cat.remove_unused_categories()
data_B.obs['nico_ct'] = data_B.obs['nico_ct'].cat.remove_unused_categories()
data_M.obs['nico_ct'] = data_M.obs['nico_ct'].cat.remove_unused_categories()

In [ ]:
data_B.obs['nico_ct'].value_counts()

In [ ]:
import scanpy as sc

In [ ]:
pg.scatter(data_M, ["louvain_labels_12"],legend_loc = "on data")

In [ ]:
counts = pd.crosstab(
    data_T.obs['louvain_labels'],
    data_T.obs['nico_ct']
)
print(counts)
proportions = counts.div(counts.sum(axis=1), axis=0)  # Normalize rows
print(proportions)

In [ ]:
prop_long = proportions.stack().reset_index()
prop_long.columns = ['louvain_labels', 'nico_ct', 'proportion']
print(prop_long.head())

In [ ]:
prop_df = (
    data_T.obs.groupby('louvain_labels')['nico_ct']
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
)

# Step 2: Identify dominant nico_ct per cluster
dominant_ct = prop_df.idxmax(axis=1)
dominant_ct_dict = dominant_ct.to_dict()

# Step 3: Add new column to adata.obs
data_T.obs['dominant_nico_ct'] = data_T.obs['louvain_labels'].map(dominant_ct_dict)

# Verify
print(data_T.obs[['louvain_labels', 'nico_ct', 'dominant_nico_ct']].head())

In [ ]:
pg.scatter(data_T, ["dominant_nico_ct","louvain_labels","nico_ct"])